# Phase 3: Conditional GAN with Self-Attention (Tasks 2 & 5)
This notebook trains a customized Conditional GAN architecture equipped with Self-Attention blocks to generate fundamental visuals (shapes) mathematically triggered by label encodings.

Constraints: Compatible with Kaggle T4 inference.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from torchvision import transforms

# Pre-requisites for custom scripts
sys.path.append(os.path.abspath('..'))
from models.cgan_attention import ConditionalGenerator, ConditionalDiscriminator
from scripts.text_processing import TextEmbedder



### Generate Synthetic Shapes Dataset
To properly train the internal CGAN quickly mapping labels to geometric traits, we generate a synthetic pool mapping text labels directly to basic shapes.

In [ ]:
class ShapesDataset(Dataset):
    def __init__(self, num_samples=25000, img_size=64):
        self.num_samples = num_samples
        self.img_size = img_size
        self.labels = ["circle", "square"] # Simplified generation pool
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

        self.data, self.texts = self._generate_data()
        
        # Initialize text embedder for CGAN conditioning (Task 3 integration)
        self.embedder = TextEmbedder() 
        
    def _generate_data(self):
        data = []
        texts = []
        for _ in range(self.num_samples):
            shape_type = np.random.choice(self.labels)
            img = Image.new('RGB', (self.img_size, self.img_size), color=(255, 255, 255))
            draw = ImageDraw.Draw(img)
            
            # draw geometries parametrically
            padding = 10
            if shape_type == "circle":
                draw.ellipse([padding, padding, self.img_size-padding, self.img_size-padding], fill="blue")
            elif shape_type == "square":
                draw.rectangle([padding, padding, self.img_size-padding, self.img_size-padding], fill="red")
            
            data.append(img)
            texts.append(shape_type)
            
        return data, texts
        
    def __len__(self):
        return self.num_samples
        
    def __getitem__(self, idx):
        img = self.transform(self.data[idx])
        text = self.texts[idx]
        
        # The returned text is naturally embedded later or directly here.
        # For dataset efficiency, we yield raw text and let collate batch processing embed it.
        return img, text
        
print("Generating Synthetic CGAN Shapes dataset...")
cgan_dataset = ShapesDataset(num_samples=2500)
print(f"{len(cgan_dataset)} specific text-to-geometric correlations constructed.")



### Training Setup (Self-Attention CGAN)
We declare the PyTorch parameters for the Kaggle-target GPU loop mapping.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modifiers
batch_size = 32
z_dim = 100
epochs = 150 # Production Training Loop enabled for Geometry Convergence
lr = 0.0002
dataloader = DataLoader(cgan_dataset, batch_size=batch_size, shuffle=True)

# Init Attention CGAN architecture (Task 5)

netG = ConditionalGenerator(z_dim=z_dim, embed_dim=512).to(device)
netD = ConditionalDiscriminator(embed_dim=512).to(device)

if torch.cuda.device_count() > 1:
    print(f"🚀 Dual-GPU Mode Active: Distributing workload across {torch.cuda.device_count()} GPUs!")
    netG = nn.DataParallel(netG)
    netD = nn.DataParallel(netD)


criterion = nn.BCELoss()
optimizerG = torch.optim.Adam(netG.parameters(), lr=lr, betas=(0.5, 0.999))
optimizerD = torch.optim.Adam(netD.parameters(), lr=lr, betas=(0.5, 0.999))

print(f"CGAN Architecture Built on {device}.")



### Training Run Pipeline

In [ ]:
# Pseudo training loop. Expose logic to user. Running requires real backend hardware (T4)
print("Starting Training Loop Segment (Task 2)...")

for epoch in range(epochs):
    for i, (imgs, texts) in enumerate(dataloader):
        
        # Phase 2 -> Embed text dynamically over batch
        with torch.no_grad():
           # Get raw embeddings from text_processing CLIP Model
           # Dimensionality formatting applied to standard sequence pooling bounds
           embeddings = cgan_dataset.embedder.get_text_embeddings(texts) 
           embeddings = embeddings.mean(dim=1).to(device) # Mean pooling down into (BATCH, EMBED_DIM)
           
        
        real_imgs = imgs.to(device)
        # Generate Target constraints
        b_size = real_imgs.size(0)
        real_label = torch.ones(b_size, device=device)
        fake_label = torch.zeros(b_size, device=device)
        
        # --- Train Discriminator ---
        optimizerD.zero_grad()
        output = netD(real_imgs, embeddings)
        d_loss_real = criterion(output, real_label)
        
        noise = torch.randn(b_size, z_dim, 1, 1, device=device)
        fake_imgs = netG(noise, embeddings)
        
        output = netD(fake_imgs.detach(), embeddings)
        d_loss_fake = criterion(output, fake_label)
        
        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        optimizerD.step()
        
        # --- Train Generator ---
        optimizerG.zero_grad()
        output = netD(fake_imgs, embeddings)
        g_loss = criterion(output, real_label)
        
        g_loss.backward()
        optimizerG.step()
        
    if epoch % 1 == 0:
        print(f"[Epoch {epoch}/{epochs}] D_Loss: {d_loss.item():.4f} G_Loss: {g_loss.item():.4f}")


if hasattr(netG, 'module'):
    torch.save(netG.module.state_dict(), 'cgan_generator.pth')
else:
    torch.save(netG.state_dict(), 'cgan_generator.pth')

print("Training Complete! Model saved as 'cgan_generator.pth'")


